# 📄 Notebook 01 — Pipeline RAG
## Sistema de Gestión de Fallas — Hidrocarburos del Perú
**Fase 2 del proyecto final · Solo hace UNA cosa: cargar PDFs, indexar en Elasticsearch y validar que el retriever funciona.**

---
### Qué hace este notebook
1. Monta Google Drive y carga credenciales
2. Carga y verifica los PDFs del corpus
3. Enriquece metadata de cada página
4. Fragmenta los documentos (chunking)
5. Genera embeddings e indexa en Elasticsearch
6. Valida con 4 tipos de consulta obligatorios

### Qué NO hace este notebook
- No crea el agente (eso es el notebook 02)
- No conecta al servidor MCP
- No genera órdenes de trabajo

> ⚠️ **Regla:** no pasar al notebook 02 hasta que los 4 tests de validación pasen correctamente.


## 📦 Celda 1 — Instalación de dependencias


In [1]:
# Celda 1 — Instalar dependencias del pipeline RAG
# elasticsearch==8.17.0 es obligatorio — el cliente 9.x
# es incompatible con Elasticsearch server 8.x
!pip install -q --upgrade \
    langchain \
    langchain-openai \
    langchain-elasticsearch \
    langchain-community \
    langchain-text-splitters \
    pypdf \
    'elasticsearch==8.17.0'

print('✅ Dependencias instaladas.')
print('   elasticsearch 8.17.0 — compatible con server 8.x')
print('   Si Colab pide reiniciar el entorno, hazlo antes de continuar.')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.2/571.2 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.1/46.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 89.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.1/65.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take 

## ☁️ Celda 2 — Montar Drive y cargar credenciales
> Toda la configuración se lee desde Google Drive. Ninguna credencial queda escrita en el notebook.


In [2]:
# Celda 2 — Montar Drive y configurar rutas
from google.colab import drive
import os

drive.mount('/content/drive')

# ── Rutas base ───────────────────────────────────────────────
BASE     = '/content/drive/MyDrive/fallas_hidrocarburos/'
BASE_PDF = BASE + 'docs_raw/'
BASE_CSV = BASE + 'csv/'
BASE_CRD = BASE + 'credenciales/'
BASE_OUT = BASE + 'outputs/'

# ── Credenciales desde Drive ──────────────────────────────────
os.environ['OPENAI_API_KEY'] = open(BASE_CRD + 'api_key.txt').read().strip()
ES_URL      = open(BASE_CRD + 'es_url.txt').read().strip()
ES_PASSWORD = open(BASE_CRD + 'es_password.txt').read().strip()
ES_USER     = 'elastic'
INDEX_NAME  = 'rag_fallas_pf_v1'   # índice nuevo — no tocar el del M4
EMBED_MODEL = 'text-embedding-3-large'

# ── Verificar que los archivos existen ────────────────────────
archivos_requeridos = [
    BASE_CRD + 'api_key.txt',
    BASE_CRD + 'es_url.txt',
    BASE_CRD + 'es_password.txt',
]
todos_ok = True
for archivo in archivos_requeridos:
    existe = os.path.exists(archivo)
    print(f"{'✅' if existe else '❌'} {archivo.split('/')[-1]}")
    if not existe: todos_ok = False

if todos_ok:
    print('\n✅ Todas las credenciales cargadas correctamente.')
else:
    print('\n❌ Faltan archivos en Drive/credenciales/. Revisalos antes de continuar.')


Mounted at /content/drive
✅ api_key.txt
✅ es_url.txt
✅ es_password.txt

✅ Todas las credenciales cargadas correctamente.


## 📂 Celda 3 — Verificar los PDFs del corpus
> Antes de cargar, verificar que los PDFs tienen texto seleccionable (no son imágenes escaneadas).


In [4]:
# Celda 3 — Verificar PDFs del corpus
import glob
from langchain_community.document_loaders import PyPDFLoader

# Listar todos los PDFs disponibles
pdfs_disponibles = sorted(glob.glob(BASE_PDF + '*.pdf'))

print(f'PDFs encontrados en Drive: {len(pdfs_disponibles)}')
print()

resumen = []
for path in pdfs_disponibles:
    nombre = path.split('/')[-1]
    try:
        loader = PyPDFLoader(path)
        pages  = loader.load()
        # Verificar que tiene texto real (no imagen escaneada)
        texto_muestra = pages[0].page_content.strip() if pages else ''
        tiene_texto   = len(texto_muestra) > 50
        estado        = '✅ OK' if tiene_texto else '⚠️  Sin texto (posible imagen escaneada)'
        print(f'{estado} | {nombre} | {len(pages)} págs | "{len(texto_muestra)} chars en pág 1"')
        resumen.append({'path': path, 'nombre': nombre,
                        'paginas': len(pages), 'ok': tiene_texto})
    except Exception as e:
        print(f'❌ ERROR | {nombre} | {str(e)}')
        resumen.append({'path': path, 'nombre': nombre,
                        'paginas': 0, 'ok': False})

pdfs_validos = [r for r in resumen if r['ok']]
print(f'\nPDFs válidos para indexar: {len(pdfs_validos)} de {len(resumen)}')

if not pdfs_validos:
    print('❌ No hay PDFs válidos. Revisar los archivos antes de continuar.')


PDFs encontrados en Drive: 5

✅ OK | ASME-B31.4.pdf | 135 págs | "223 chars en pág 1"
✅ OK | ASME-B31.8.pdf | 194 págs | "569 chars en pág 1"
✅ OK | DS_018-2004-EM.pdf | 34 págs | "2114 chars en pág 1"
✅ OK | DS_081-2007-EM.pdf | 133 págs | "2231 chars en pág 1"
✅ OK | historial_fallas_SAP.pdf | 4 págs | "1277 chars en pág 1"

PDFs válidos para indexar: 5 de 5


## 📖 Celda 4 — Cargar todos los PDFs
> Se cargan todos los PDFs válidos y se construye el corpus completo.


In [5]:
# Celda 4 — Cargar todos los PDFs del corpus
from langchain_community.document_loaders import PyPDFLoader

corpus_docs = []
errores     = []

for r in pdfs_validos:
    try:
        loader = PyPDFLoader(r['path'])
        pages  = loader.load()
        corpus_docs.extend(pages)
        print(f'  ✅ {r["nombre"]:40s} → {len(pages):3d} páginas')
    except Exception as e:
        errores.append(r['nombre'])
        print(f'  ❌ {r["nombre"]:40s} → ERROR: {e}')

print(f'\nTotal páginas cargadas : {len(corpus_docs)}')
print(f'Documentos con error   : {len(errores)}')

if errores:
    print(f'Errores en: {errores}')

# Mostrar muestra del primer documento
if corpus_docs:
    print(f'\nEjemplo — página 1 del primer doc:')
    print(f'  Source : {corpus_docs[0].metadata["source"].split("/")[-1]}')
    print(f'  Página : {corpus_docs[0].metadata.get("page", "N/A")}')
    print(f'  Texto  : {corpus_docs[0].page_content[:300]}...')


  ✅ ASME-B31.4.pdf                           → 135 páginas
  ✅ ASME-B31.8.pdf                           → 194 páginas
  ✅ DS_018-2004-EM.pdf                       →  34 páginas
  ✅ DS_081-2007-EM.pdf                       → 133 páginas
  ✅ historial_fallas_SAP.pdf                 →   4 páginas

Total páginas cargadas : 500
Documentos con error   : 0

Ejemplo — página 1 del primer doc:
  Source : ASME-B31.4.pdf
  Página : 0
  Texto  : (Revisión  de  ASME  B31.4­2019)
Código  ASME  para  tuberías  a  presión,  B31
UN  CÓDIGO  INTERNACIONAL  DE  TUBERÍAS  ®
Sistemas  para  
Líquidos  y  Lodos
Transporte
Tubería
ASME  B31.4­2022
Machine Translated by Google...


## 🏷️ Celda 5 — Enriquecimiento de metadata
> Cada página recibe metadata adicional que el agente usará para citar la fuente correctamente.


In [6]:
# Celda 5 — Enriquecer metadata de cada documento
# Ajustado para los 5 PDFs disponibles:
#   DS_081-2007-EM.pdf, DS_018-2004-EM.pdf,
#   ASME-B31.4.pdf, ASME-B31.8.pdf, historial_fallas_SAP.pdf
import re
from langchain_core.documents import Document

# Mapeo exacto de nombres de archivo a categorías
# Clave = parte del nombre del archivo (case-insensitive)
CATEGORIA_MAP = {
    'historial_fallas_SAP' : 'historial_operativo',
    'DS_081'               : 'normativa_peruana',      # Reglamento de Transporte
    'DS_018'               : 'normativa_peruana',      # Norma de Servicio de Transporte
    'ASME-B31.4'           : 'norma_tecnica_internacional',  # Tuberias liquidos
    'ASME-B31.8'           : 'norma_tecnica_internacional',  # Gasoductos
    'ASME_B31'             : 'norma_tecnica_internacional',  # fallback ASME generico
}

# Patrones para extraer entidades del texto
_RE_EQUIPO   = re.compile(r'\b(EQ[A-Z]-\d{4})\b', re.IGNORECASE)
_RE_FECHA    = re.compile(r'\b(\d{2}/\d{2}/\d{4})\b')
_RE_ARTICULO = re.compile(r'[Aa]rt[íi]culo\s+(\d+)', re.IGNORECASE)

def detectar_categoria(nombre_archivo):
    """Detecta la categoría del documento según su nombre de archivo."""
    for clave, cat in CATEGORIA_MAP.items():
        if clave.lower() in nombre_archivo.lower():
            return cat
    return 'documento_tecnico'  # fallback si no coincide ninguna clave

def enriquecer_documento(doc):
    """Agrega metadata enriquecida a cada página del corpus."""
    texto  = doc.page_content
    fuente = doc.metadata.get('source', '')
    nombre = fuente.split('/')[-1].replace('.pdf', '')

    # Extraer entidades del texto
    equipos   = _RE_EQUIPO.findall(texto)
    fechas    = _RE_FECHA.findall(texto)
    articulos = _RE_ARTICULO.findall(texto)

    metadata_enriquecida = {
        # Metadata original preservada
        'source'       : fuente,
        'page'         : doc.metadata.get('page', 0),
        # Metadata nueva para el agente
        'doc_name'     : nombre,
        'categoria'    : detectar_categoria(nombre),
        'equipos_ref'  : ', '.join(set(equipos))   if equipos   else 'N/A',
        'fecha_ref'    : fechas[0]                  if fechas    else 'N/A',
        'articulos_ref': ', '.join(set(articulos)) if articulos else 'N/A',
        'tiene_equipo' : len(equipos)   > 0,
        'tiene_norma'  : len(articulos) > 0,
    }
    return Document(page_content=texto, metadata=metadata_enriquecida)

# Aplicar enriquecimiento a todo el corpus
corpus_enriquecido = [enriquecer_documento(doc) for doc in corpus_docs]

print(f'✅ Corpus enriquecido: {len(corpus_enriquecido)} páginas')

# Verificar que los 5 documentos se mapearon correctamente
from collections import Counter
print(f'\nVerificación de categorías por documento:')
docs_por_nombre = {}
for d in corpus_enriquecido:
    n = d.metadata['doc_name']
    c = d.metadata['categoria']
    if n not in docs_por_nombre:
        docs_por_nombre[n] = c

for nombre, cat in sorted(docs_por_nombre.items()):
    icono = '✅' if cat != 'documento_tecnico' else '⚠️ '
    print(f'  {icono} {nombre:35s} → {cat}')

# Estadísticas por categoría
categorias = Counter(d.metadata['categoria'] for d in corpus_enriquecido)
print(f'\nTotal páginas por categoría:')
for cat, count in categorias.most_common():
    print(f'  {cat:40s}: {count} páginas')

# Alerta si algún doc cayó en 'documento_tecnico' (no fue reconocido)
no_reconocidos = [n for n,c in docs_por_nombre.items() if c == 'documento_tecnico']
if no_reconocidos:
    print(f'\n⚠️  Documentos no reconocidos en CATEGORIA_MAP: {no_reconocidos}')
    print('   Agregar su nombre clave al diccionario CATEGORIA_MAP.')
else:
    print(f'\n✅ Todos los documentos fueron categorizados correctamente.')


✅ Corpus enriquecido: 500 páginas

Verificación de categorías por documento:
  ✅ ASME-B31.4                          → norma_tecnica_internacional
  ✅ ASME-B31.8                          → norma_tecnica_internacional
  ✅ DS_018-2004-EM                      → normativa_peruana
  ✅ DS_081-2007-EM                      → normativa_peruana
  ✅ historial_fallas_SAP                → historial_operativo

Total páginas por categoría:
  norma_tecnica_internacional             : 329 páginas
  normativa_peruana                       : 167 páginas
  historial_operativo                     : 4 páginas

✅ Todos los documentos fueron categorizados correctamente.


## ✂️ Celda 6 — Fragmentación (chunking)
> Dividir los documentos en fragmentos del tamaño correcto para el RAG. El tamaño 800/100 es óptimo para documentos normativos con párrafos largos.


In [7]:
# Celda 6 — Fragmentación del corpus
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Parámetros justificados:
# chunk_size=800   → documentos normativos tienen párrafos largos (150-300 palabras)
#                    chunk_size 500 cortaría los artículos a la mitad
# chunk_overlap=100 → 12.5% de overlap para no perder contexto en los bordes
# separators       → prioriza cortar por párrafo > línea > punto > espacio
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=['\n\n', '\n', '. ', ' ', ''],
    length_function=len,
)

splits = splitter.split_documents(corpus_enriquecido)

print(f'✅ Fragmentación completada')
print(f'   Documentos originales : {len(corpus_enriquecido)}')
print(f'   Chunks generados      : {len(splits)}')
print(f'   Promedio chars/chunk  : {sum(len(s.page_content) for s in splits) // len(splits)}')
print(f'   Chunk más corto       : {min(len(s.page_content) for s in splits)} chars')
print(f'   Chunk más largo       : {max(len(s.page_content) for s in splits)} chars')

# Distribución de chunks por documento
from collections import Counter
chunks_por_doc = Counter(s.metadata['doc_name'] for s in splits)
print(f'\nChunks por documento:')
for doc, count in chunks_por_doc.most_common():
    print(f'  {doc:45s}: {count:3d} chunks')

# Verificar que la metadata se preservó
print(f'\nEjemplo chunk 0:')
print(f'  doc_name : {splits[0].metadata["doc_name"]}')
print(f'  categoria: {splits[0].metadata["categoria"]}')
print(f'  texto    : {splits[0].page_content[:200]}...')


✅ Fragmentación completada
   Documentos originales : 500
   Chunks generados      : 2665
   Promedio chars/chunk  : 704
   Chunk más corto       : 20 chars
   Chunk más largo       : 800 chars

Chunks por documento:
  ASME-B31.8                                   : 1081 chunks
  ASME-B31.4                                   : 973 chunks
  DS_081-2007-EM                               : 474 chunks
  DS_018-2004-EM                               : 129 chunks
  historial_fallas_SAP                         :   8 chunks

Ejemplo chunk 0:
  doc_name : ASME-B31.4
  categoria: norma_tecnica_internacional
  texto    : (Revisión  de  ASME  B31.4­2019)
Código  ASME  para  tuberías  a  presión,  B31
UN  CÓDIGO  INTERNACIONAL  DE  TUBERÍAS  ®
Sistemas  para  
Líquidos  y  Lodos
Transporte
Tubería
ASME  B31.4­2022
Machi...


## 🔌 Celda 7 — Conectar a Elasticsearch y preparar el índice
> Se conecta al cluster de Elasticsearch y se elimina el índice anterior si existe para hacer una indexación limpia.


In [8]:
# Celda 7 — Conectar a Elasticsearch
from langchain_openai import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore

# Inicializar el modelo de embeddings
embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
print(f'✅ Modelo de embeddings: {EMBED_MODEL}')

# Conectar al vector store
vector_store = ElasticsearchStore(
    index_name=INDEX_NAME,
    embedding=embeddings,
    es_url=ES_URL,
    es_user=ES_USER,
    es_password=ES_PASSWORD,
)

# Verificar conexión con el cluster
try:
    info = vector_store.client.info()
    print(f'✅ Elasticsearch conectado')
    print(f'   Cluster : {info["cluster_name"]}')
    print(f'   Versión : {info["version"]["number"]}')
except Exception as e:
    print(f'❌ Error de conexión: {e}')
    print('   Verificar ES_URL y ES_PASSWORD en Drive/credenciales/')

# Verificar si el índice ya existe
existe = vector_store.client.indices.exists(index=INDEX_NAME)
print(f'\nÍndice "{INDEX_NAME}" existe: {existe}')

if existe:
    count = vector_store.client.count(index=INDEX_NAME)
    print(f'   Chunks actuales en el índice: {count["count"]}')
    print(f'   Se eliminará para re-indexar limpio.')


✅ Modelo de embeddings: text-embedding-3-large
✅ Elasticsearch conectado
   Cluster : elasticsearch
   Versión : 8.19.19

Índice "rag_fallas_pf_v1" existe: False


## 🚀 Celda 8 — Indexar en Elasticsearch
> Esta celda genera los embeddings y los almacena. Es la más costosa en tiempo (~2-5 min) y en tokens de OpenAI.


In [10]:
# Celda 8 — Indexación en Elasticsearch
# ⚠️ Esta celda consume tokens de OpenAI — ejecutar solo cuando sea necesario
# ⚠️ Ejecutar solo UNA vez por sesión

CONFIRMAR_INDEXACION = True  # Cambiar a False para saltarse esta celda

if not CONFIRMAR_INDEXACION:
    print('Indexación omitida. Cambiar CONFIRMAR_INDEXACION=True para ejecutar.')
else:
    # 1. Eliminar índice previo si existe
    if vector_store.client.indices.exists(index=INDEX_NAME):
        vector_store.client.indices.delete(index=INDEX_NAME)
        print(f'🗑️  Índice "{INDEX_NAME}" eliminado para re-indexación limpia.')

    # 2. Preparar IDs únicos para cada chunk
    ids = [f'chunk_{i:04d}' for i in range(len(splits))]

    # 3. Indexar en lotes para no saturar la API (y también la API de OpenAI)
    print(f'⏳ Indexando {len(splits)} chunks en Elasticsearch...')
    print(f'   Esto puede tardar 2-5 minutos según el tamaño del corpus.')

    # Definir un tamaño de lote para los documentos que se envían a la API de OpenAI
    # Un promedio de 176 tokens/chunk (704 chars/chunk) significa que 500 chunks
    # son aproximadamente 88,000 tokens, lo cual está por debajo del límite de 300,000.
    batch_size_for_openai = 500
    inserted_ids_total = []

    for i in range(0, len(splits), batch_size_for_openai):
        batch_splits = splits[i:i + batch_size_for_openai]
        batch_ids = ids[i:i + batch_size_for_openai]
        print(f"   Procesando lote {i // batch_size_for_openai + 1}/{(len(splits) + batch_size_for_openai - 1) // batch_size_for_openai} ({len(batch_splits)} chunks)...")
        inserted_ids = vector_store.add_documents(
            documents=batch_splits,
            ids=batch_ids,
            bulk_kwargs={'chunk_size': 50}  # lotes de 50 para no saturar Elasticsearch
        )
        inserted_ids_total.extend(inserted_ids)

    # 4. Refrescar el índice para que las búsquedas sean inmediatas
    vector_store.client.indices.refresh(index=INDEX_NAME)

    # 5. Verificar la indexación
    count_final = vector_store.client.count(index=INDEX_NAME)
    print(f'\n✅ Indexación completada')
    print(f'   Chunks solicitados : {len(splits)}')
    print(f'   Chunks indexados   : {count_final["count"]}')

    if count_final['count'] == len(splits):
        print(f'   ✅ Conteo correcto — todos los chunks fueron indexados')
    else:
        print(f'   ⚠️  Diferencia en conteo — verificar si hubo errores')


🗑️  Índice "rag_fallas_pf_v1" eliminado para re-indexación limpia.
⏳ Indexando 2665 chunks en Elasticsearch...
   Esto puede tardar 2-5 minutos según el tamaño del corpus.
   Procesando lote 1/6 (500 chunks)...
   Procesando lote 2/6 (500 chunks)...
   Procesando lote 3/6 (500 chunks)...
   Procesando lote 4/6 (500 chunks)...
   Procesando lote 5/6 (500 chunks)...
   Procesando lote 6/6 (165 chunks)...

✅ Indexación completada
   Chunks solicitados : 2665
   Chunks indexados   : 2665
   ✅ Conteo correcto — todos los chunks fueron indexados


## 🧪 Celda 9 — Validación: 4 tipos de consulta obligatorios
> Esta es la celda más importante. El RAG está listo para el siguiente paso SOLO si los 4 tests pasan.


In [12]:
# Celda 9 — Validación de los 4 tipos de consulta

def test_rag(query, descripcion, score_minimo=0.70, k=4):
    '"""Ejecuta una consulta RAG y muestra los resultados con diagnóstico."""'
    print(f'\n{"─"*60}')
    print(f'Test: {descripcion}')
    print(f'Query: "{query}"')
    print(f'{"─"*60}')

    results = vector_store.similarity_search_with_score(query, k=k)

    if not results:
        print('  ⚠️  Sin resultados en el índice.')
        return False

    tiene_relevante = False
    for doc, score in results:
        relevante = score >= score_minimo
        if relevante: tiene_relevante = True
        icono = '✅' if relevante else '⚠️ '
        print(f'  {icono} Score: {score:.3f} | "{doc.metadata.get("doc_name","?"):30s} | Pág: {doc.metadata.get("page","?"):3}"')
        if relevante:
            print(f'     Texto: {doc.page_content[:200]}...')

    return tiene_relevante

print('=' * 60)
print('VALIDACIÓN DEL PIPELINE RAG')
print('=' * 60)

# ── TEST 1: Consulta cubierta en el corpus ────────────────────
ok1 = test_rag(
    query='presión máxima de operación en gasoductos',
    descripcion='TEST 1 — Consulta cubierta por normativa',
    score_minimo=0.70
)

# ── TEST 2: Consulta sobre historial de fallas ────────────────
ok2 = test_rag(
    query='falla de vibración excesiva en compresor solución',
    descripcion='TEST 2 — Consulta sobre historial SAP',
    score_minimo=0.70
)

# ── TEST 3: Consulta que cruza dos documentos ─────────────────
ok3 = test_rag(
    query='reemplazo de sello mecánico bomba centrifuga procedimiento',
    descripcion='TEST 3 — Consulta que cruza historial y norma técnica',
    score_minimo=0.65  # más bajo porque cruza documentos
)

# ── TEST 4: Consulta fuera del corpus ─────────────────────────
results_fuera = vector_store.similarity_search_with_score(
    'receta de cocina peruana ceviche',
    k=3
)
score_max_fuera = max(s for _, s in results_fuera) if results_fuera else 0
ok4 = score_max_fuera < 0.70  # debe tener score BAJO
print(f'\n{"─"*60}')
print(f'Test: TEST 4 — Consulta completamente fuera del dominio')
print(f'Query: "receta de cocina peruana ceviche"')
print(f'{"─"*60}')
print(f'  Score máximo encontrado: {score_max_fuera:.3f}')
print(f'  {'✅ Correcto — score bajo, fuera del dominio' if ok4 else '❌ Problema — score alto en consulta fuera del dominio'}')

# ── RESULTADO FINAL ───────────────────────────────────────────
print(f'\n{"="*60}')
print('RESULTADO FINAL DE LA VALIDACIÓN')
print(f'{"="*60}')
tests = [
    ('TEST 1 - Consulta normativa', ok1),
    ('TEST 2 - Historial de fallas', ok2),
    ('TEST 3 - Consulta cruzada', ok3),
    ('TEST 4 - Fuera del dominio', ok4),
]
todos_ok = True
for nombre, resultado in tests:
    icono = '✅' if resultado else '❌'
    print(f'  {icono} {nombre}')
    if not resultado: todos_ok = False

print()
if todos_ok:
    print('✅ TODOS LOS TESTS PASAN — El RAG está listo.')
    print('   Puedes continuar con el notebook 02 (agente LangChain).')
else:
    print('❌ ALGUNOS TESTS FALLARON — Revisar antes de continuar.')
    print('   Ver sección de diagnóstico en la celda 10.')


VALIDACIÓN DEL PIPELINE RAG

────────────────────────────────────────────────────────────
Test: TEST 1 — Consulta cubierta por normativa
Query: "presión máxima de operación en gasoductos"
────────────────────────────────────────────────────────────
  ✅ Score: 0.817 | "DS_081-2007-EM                 | Pág: 101"
     Texto: longitudinal o transversal excesiva o la desviación de la tubería, el Operador 
deberá implementar un programa de monitoreo y evaluación que incluya 
criterios de corrección con la finalidad de preven...
  ✅ Score: 0.815 | "ASME-B31.4                     | Pág:  89"
     Texto: deseada  en  estado  estable  aumentada  de  acuerdo  con  el  requisito  de  
diseño  de  este  Código,  y  el  sistema  ha  sido  probado  previamente  por  
una  duración  y  a  una  presión  igual...
  ✅ Score: 0.800 | "DS_081-2007-EM                 | Pág:  75"
     Texto: c. Facilidades ocupadas por personas que se encuentran confinadas, o de 
movilidad restringida, o con dificultad para 

## 🔧 Celda 10 — Diagnóstico si algo falla
> Ejecutar solo si algún test de la celda 9 falló. Ayuda a identificar el problema exacto.


In [ ]:
# Celda 10 — Diagnóstico de problemas (ejecutar solo si algo falló)

print('DIAGNÓSTICO DEL PIPELINE RAG')
print('=' * 50)

# 1. Verificar que el índice existe y tiene chunks
print('\n1. Estado del índice:')
existe = vector_store.client.indices.exists(index=INDEX_NAME)
print(f'   Existe: {existe}')
if existe:
    count = vector_store.client.count(index=INDEX_NAME)
    print(f'   Chunks: {count["count"]}')

# 2. Verificar que los chunks tienen metadata correcta
print('\n2. Muestra de chunks indexados:')
resp = vector_store.client.search(
    index=INDEX_NAME,
    body={'query': {'match_all': {}}, 'size': 3}
)
for hit in resp['hits']['hits']:
    src = hit['_source'].get('metadata', {})
    print(f'  ID: {hit["_id"]} | doc: {src.get("doc_name","?")} | '",
          f'"pág: {src.get("page","?")} | cat: {src.get("categoria","?")}')

# 3. Probar embedding directo
print('\n3. Test de embedding directo:')
try:
    vec = embeddings.embed_query('presión en gasoductos')
    print(f'   ✅ Embedding OK — dimensión: {len(vec)}')
except Exception as e:
    print(f'   ❌ Error en embedding: {e}')
    print('   → Verificar OPENAI_API_KEY')

# 4. Búsqueda directa en Elasticsearch
print('\n4. Búsqueda de texto simple (sin embedding):')
resp_texto = vector_store.client.search(
    index=INDEX_NAME,
    body={'query': {'match': {'text': 'presión'}}, 'size': 3}
)
hits = resp_texto['hits']['hits']
print(f'   Resultados: {len(hits)}')
for h in hits:
    print(f'  Score: {h["_score"]:.3f} | {h["_source"].get("metadata",{}).get("doc_name","?")}')

# 5. Causas comunes y soluciones
print('\n5. Causas comunes de fallos:')
print('   ❌ Score bajo en TEST 1/2/3:')
print('      → Los PDFs pueden ser imágenes escaneadas')
print('      → chunk_size demasiado grande o pequeño')
print('      → El documento relevante no está en el corpus')
print('   ❌ Score alto en TEST 4 (fuera de dominio):')
print('      → El corpus tiene documentos muy genéricos')
print('      → Aumentar el score mínimo en el retriever a 0.75')


## 💾 Celda 11 — Guardar el notebook en Drive
> Guardar una copia del notebook ejecutado en Drive para tener evidencia del pipeline funcionando.


In [13]:
# Celda 11 — Guardar resumen de la indexación en Drive
import json
from datetime import datetime

resumen_indexacion = {
    'fecha'            : datetime.now().isoformat(),
    'index_name'       : INDEX_NAME,
    'embed_model'      : EMBED_MODEL,
    'chunk_size'       : CHUNK_SIZE,
    'chunk_overlap'    : CHUNK_OVERLAP,
    'total_paginas'    : len(corpus_docs),
    'total_chunks'     : len(splits),
    'documentos'       : [r['nombre'] for r in pdfs_validos],
    'chunks_por_doc'   : dict(chunks_por_doc),
    'tests_validacion' : {
        'test1_normativa'   : bool(ok1),
        'test2_historial'   : bool(ok2),
        'test3_cruzado'     : bool(ok3),
        'test4_fuera_dominio': bool(ok4),
    }
}

# Guardar en Drive
ruta_resumen = BASE_OUT + 'rag_indexacion_resumen.json'
os.makedirs(BASE_OUT, exist_ok=True)
with open(ruta_resumen, 'w', encoding='utf-8') as f:
    json.dump(resumen_indexacion, f, indent=2, ensure_ascii=False)

print(f'✅ Resumen guardado en: {ruta_resumen}')
print()
print(json.dumps(resumen_indexacion, indent=2, ensure_ascii=False))

print('\n' + '='*60)
print('PASO 2 COMPLETADO')
print('='*60)
print('El pipeline RAG está listo.')
print('Próximo paso: Notebook 02 — Agente LangChain.')
print()
print('Antes de cerrar este notebook:')
print('  File → Download → Download .ipynb')
print('  Mover a: backend/notebooks/01_rag_indexacion.ipynb')
print('  git add . && git commit -m "feat: notebook RAG completado"')


✅ Resumen guardado en: /content/drive/MyDrive/fallas_hidrocarburos/outputs/rag_indexacion_resumen.json

{
  "fecha": "2026-08-09T04:40:32.618464",
  "index_name": "rag_fallas_pf_v1",
  "embed_model": "text-embedding-3-large",
  "chunk_size": 800,
  "chunk_overlap": 100,
  "total_paginas": 500,
  "total_chunks": 2665,
  "documentos": [
    "ASME-B31.4.pdf",
    "ASME-B31.8.pdf",
    "DS_018-2004-EM.pdf",
    "DS_081-2007-EM.pdf",
    "historial_fallas_SAP.pdf"
  ],
  "chunks_por_doc": {
    "ASME-B31.4": 973,
    "ASME-B31.8": 1081,
    "DS_018-2004-EM": 129,
    "DS_081-2007-EM": 474,
    "historial_fallas_SAP": 8
  },
  "tests_validacion": {
    "test1_normativa": true,
    "test2_historial": true,
    "test3_cruzado": true,
    "test4_fuera_dominio": true
  }
}

PASO 2 COMPLETADO
El pipeline RAG está listo.
Próximo paso: Notebook 02 — Agente LangChain.

Antes de cerrar este notebook:
  File → Download → Download .ipynb
  Mover a: backend/notebooks/01_rag_indexacion.ipynb
  git add . 